# 5주차. Supervisor/Sub-Agent 위임과 Golden Scenario 하네스

## 0. 목표

이번 주에는 supervisor가 직접 일하지 않고, 요청 성격에 맞는 나나 또는 카나 sub-agent에게 위임하는 흐름을 검증한다.

성취 기준:

- supervisor와 sub-agent의 역할 차이를 설명할 수 있다.
- `selected_agent`와 내부 tool trace를 함께 확인할 수 있다.
- Golden Scenario가 라우팅 검증에 필요한 이유를 설명할 수 있다.

## 1. 준비

로컬 `Jupyter Notebook` 또는 `JupyterLab`에서 repo 루트를 기준으로 실행한다.

모든 실습은 실제 OpenAI API를 호출한다. API 키와 모델 설정은 repo 루트의 `.env`에서 읽는다.

처음 읽는 순서: `docs/orientation.md` → 이 노트북 → `week05.py`의 `# TODO 문제`를 먼저 풀고 바로 아래 `# 모범 답안`과 비교한 뒤 Gradio 데모를 실행한다.


In [ ]:
# 흐름: repo 설정과 공통 helper를 준비한다.
# 5주차의 핵심은 supervisor trace와 sub-agent 내부 trace를 나눠 보는 것이다.
import json
import os
import sys
from pathlib import Path
from typing import Any


def find_repo_root(start: Path) -> Path:
    # 현재 폴더부터 부모 폴더로 올라가며 repo 루트를 찾는다.
    # 노트북을 notebook/ 안에서 실행해도, repo 루트에서 실행해도 같은 경로를 쓰기 위한 장치다.
    for candidate in [start, *start.parents]:
        if (candidate / ".env").exists() or (candidate / ".env.example").exists():
            return candidate
    raise RuntimeError("repo root를 찾지 못했습니다. repo 안에서 노트북을 실행하세요.")


# 앞으로 나오는 상대 경로는 모두 이 repo 루트를 기준으로 맞춘다.
REPO_ROOT = find_repo_root(Path.cwd())

if str(REPO_ROOT) not in sys.path:
    # weekXX.py 같은 repo 루트의 Python 파일을 import할 수 있게 경로를 추가한다.
    sys.path.insert(0, str(REPO_ROOT))
# 파일 저장/DB 생성 위치가 흔들리지 않도록 작업 폴더를 repo 루트로 고정한다.
os.chdir(REPO_ROOT)

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_openai import ChatOpenAI

# override=True를 사용해 노트북 커널에 남아 있던 이전 환경 변수보다 repo .env를 우선한다.
load_dotenv(REPO_ROOT / ".env", override=True)

# 모델 이름은 .env에서 바꿀 수 있고, 값이 없으면 수업 기본값을 사용한다.
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-4o-mini")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

if not os.getenv("OPENAI_API_KEY"):
    # API key가 없으면 모델 호출 오류가 길게 나오므로, 준비 셀에서 바로 멈춘다.
    raise RuntimeError("repo 루트 .env 파일에 OPENAI_API_KEY를 설정한 뒤 다시 실행하세요.")


def make_model(max_tokens: int = 500) -> ChatOpenAI:
    # temperature=0은 같은 입력에서 비슷한 tool 선택/구조화 결과가 나오게 하기 위한 설정이다.
    return ChatOpenAI(model=OPENAI_MODEL, temperature=0, max_completion_tokens=max_tokens)


def show_json(value: Any) -> None:
    # ensure_ascii=False로 한글 payload를 사람이 읽기 쉬운 그대로 출력한다.
    print(json.dumps(value, ensure_ascii=False, indent=2, default=str))


def final_text(agent_result: dict[str, Any]) -> str:
    # agent 실행 결과에서 마지막 message가 사용자에게 보이는 최종 답변이다.
    return agent_result["messages"][-1].content


def extract_tool_trace(agent_result: dict[str, Any]) -> list[dict[str, Any]]:
    # LangChain message 전체를 그대로 보여주면 복잡하므로, 수업에서 볼 핵심만 뽑는다.
    trace = []
    for message in agent_result.get("messages", []):
        # tool_calls는 모델이 "이 도구를 이 인자로 실행해줘"라고 요청한 기록이다.
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            # type == "tool"인 message는 실제 tool 실행 결과다.
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return trace


## 2. 개념

오늘의 큰 그림: Sub-agent는 역할과 사용할 도구를 나눈 agent다. Supervisor는 직접 업무 도구를 들고 있지 않고, `nana_agent`, `kana_agent`처럼 sub-agent를 감싼 tool 중 하나를 호출한다.

반드시 이해할 것:

- supervisor trace는 어떤 sub-agent에게 위임했는지 보여준다.
- sub-agent 내부 trace는 실제 업무 tool이 실행됐는지 보여준다.
- Golden Scenario는 라우팅을 반복 가능한 케이스로 검증하는 장치다.

지금은 몰라도 되는 것:

- 대규모 multi-agent orchestration 패턴
- agent 간 장기 memory 공유 설계
- skills/MD를 production 정책으로 관리하는 방식

막혔을 때 볼 trace: supervisor의 `nana_agent`/`kana_agent` tool call, delegate payload 안의 `trace`, `inner_tool_names`.


## 3. 기본 개념 실습

가장 작은 흐름은 supervisor가 개인 요청은 나나에게, 그룹 조율 요청은 카나에게 위임하는 것이다.


In [ ]:
# 흐름: Nana/Kana가 실제 일을 처리할 내부 tool과 sub-agent를 만든다.
# supervisor는 이 업무 tool을 직접 갖지 않고 sub-agent 위임 tool만 갖는다.
# Nana 내부에서만 쓰는 업무 tool이다. supervisor에게 직접 주지 않는다.
@tool("personal_create_schedule", description="나나가 개인 일정 초안을 만든다.")
def personal_create_schedule(title: str, date: str, start_time: str) -> str:
    """Create a personal schedule."""
    # sub-agent tool도 실행 결과를 JSON 문자열 payload로 남긴다.
    return json.dumps({"ok": True, "schedule": {"title": title, "date": date, "start_time": start_time}}, ensure_ascii=False)

# Kana 내부에서만 쓰는 그룹 일정 확정 tool이다.
@tool("group_confirm_slot", description="카나가 멤버 응답을 근거로 그룹 시간을 확정한다.")
def group_confirm_slot(topic: str, selected_slot: str, members: list[str], reason: str) -> str:
    """Confirm a group slot from member replies."""
    return json.dumps({"ok": True, "topic": topic, "selected_slot": selected_slot, "members": members, "reason": reason}, ensure_ascii=False)
# sub-agent는 자기 역할에 필요한 tool만 가진다.
nana_agent = create_agent(
    model=make_model(600),
    tools=[personal_create_schedule],
    system_prompt="너는 개인 메이트 나나다. 오늘은 2026-04-23이다. 상대 날짜는 이 날짜 기준으로 YYYY-MM-DD로 바꾼다. 개인 일정 요청은 personal_create_schedule 도구를 호출한다.",
)

kana_agent = create_agent(
    model=make_model(700),
    tools=[group_confirm_slot],
    system_prompt="너는 그룹 메이트 카나다. 멤버 응답에서 모두 가능한 시간을 찾으면 group_confirm_slot 도구를 호출한다.",
)

# 여기서부터는 supervisor가 호출할 위임 tool이다.
@tool("nana_agent", description="개인 일정 요청을 나나 sub-agent에게 위임한다.")
def delegate_to_nana(request: str) -> str:
    """Delegate a personal request to Nana sub-agent."""
    # 위임 tool 내부에서 실제 Nana agent를 한 번 더 실행한다.
    agent_result = nana_agent.invoke({"messages": [{"role": "user", "content": request}]})
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    # 내부 trace까지 payload에 넣어야 supervisor 밖에서 sub-agent 실행을 설명할 수 있다.
    return json.dumps({"agent": "nana", "answer": agent_result["messages"][-1].content, "trace": trace}, ensure_ascii=False)

@tool("kana_agent", description="그룹 일정 조율 요청과 멤버 응답을 카나 sub-agent에게 위임한다.")
def delegate_to_kana(request: str, member_replies: str) -> str:
    """Delegate a group coordination request to Kana sub-agent."""
    message = f"요청: {request}\n멤버 응답:\n{member_replies}"
    agent_result = kana_agent.invoke({"messages": [{"role": "user", "content": message}]})
    trace = []
    for message in agent_result.get("messages", []):
        for call in getattr(message, "tool_calls", []) or []:
            trace.append({"event": "tool_call", "tool_name": call.get("name"), "arguments": call.get("args", {})})
        if getattr(message, "type", None) == "tool":
            trace.append({"event": "tool_result", "tool_name": getattr(message, "name", None), "content": message.content})
    return json.dumps({"agent": "kana", "answer": agent_result["messages"][-1].content, "trace": trace}, ensure_ascii=False)

# supervisor는 업무를 직접 하지 않고 nana_agent/kana_agent 중 어디로 보낼지만 결정한다.
supervisor = create_agent(
    model=make_model(900),
    tools=[delegate_to_nana, delegate_to_kana],
    system_prompt="너는 카나메이트 supervisor다. 개인 일정 요청은 nana_agent tool을 호출하고, 그룹 일정 조율 요청은 kana_agent tool을 호출한다. 직접 처리하지 말고 반드시 적절한 sub-agent tool을 호출한 뒤 그 결과를 수강생에게 요약한다.",
)

def delegated_agent_from_trace(agent_result: dict[str, Any]) -> str:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_call" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return event["tool_name"].replace("_agent", "")
    return "unknown"


def delegated_payload_from_trace(agent_result: dict[str, Any]) -> dict[str, Any]:
    for event in extract_tool_trace(agent_result):
        if event.get("event") == "tool_result" and event.get("tool_name") in {"nana_agent", "kana_agent"}:
            return json.loads(event["content"])
    return {}


## 4. 카나메이트 확장 예제

라우팅을 눈으로 한 번만 확인하지 않고 `run_supervisor_case` 하네스로 감싼다. 이 하네스는 선택된 sub-agent와 내부 tool trace를 한 report로 만든다.


In [ ]:
# 흐름: 하나의 golden case를 실행해 선택 agent와 내부 tool 이름을 리포트로 만든다.
# selected_agent만 보지 말고 inner_tool_names까지 확인해야 라우팅 성공이다.
def run_supervisor_case(case: dict[str, Any]) -> dict[str, Any]:
    """Run one routing case and return a compact report."""
    # 멤버 응답이 없는 개인 요청은 request만 supervisor에게 보낸다.
    content = case["request"]
    if case.get("member_replies"):
        content = f"요청: {case['request']}\n멤버 응답:\n{case['member_replies']}"

    result = supervisor.invoke({"messages": [{"role": "user", "content": content}]})
    selected_agent = delegated_agent_from_trace(result)
    delegate_payload = delegated_payload_from_trace(result)
    # delegate_payload 안의 trace를 봐야 sub-agent가 실제 업무 tool을 불렀는지 확인할 수 있다.
    inner_tool_names = [
        event["tool_name"]
        for event in delegate_payload.get("trace", [])
        if event.get("event") == "tool_call"
    ]
    return {
        "name": case["name"],
        "expected_agent": case.get("expected_agent"),
        "selected_agent": selected_agent,
        "expected_inner_tool": case.get("expected_inner_tool"),
        "inner_tool_names": inner_tool_names,
        "answer": final_text(result),
        "trace": extract_tool_trace(result),
        "delegate_payload": delegate_payload,
    }

extension_case = {
    "name": "group_slot",
    "request": "팀 회의 시간을 조율해줘",
    "member_replies": "민수: 2026-04-24 10:00 가능\n지아: 2026-04-24 10:00 가능\n준호: 2026-04-24 10:00 가능",
    "expected_agent": "kana",
    "expected_inner_tool": "group_confirm_slot",
}
extension_result = run_supervisor_case(extension_case)

show_json({"selected_agent": extension_result["selected_agent"], "inner_tool_names": extension_result["inner_tool_names"]})
print(extension_result["answer"])
show_json(extension_result["trace"])


## 5. 확장 예제 실행

같은 하네스로 개인 일정 케이스와 그룹 조율 케이스를 연속 실행한다. 수강생은 케이스 입력을 바꿔 라우팅 결과와 내부 tool 이름을 비교한다.


In [ ]:
# 흐름: 여러 golden case를 반복 실행해 supervisor 라우팅이 흔들리지 않는지 확인한다.
# assert는 기대 agent와 실제 내부 tool이 모두 맞는지 점검한다.
extension_cases = [
    {
        "name": "personal_schedule",
        "request": "내일 9시에 민수와 1:1 일정 잡아줘",
        "member_replies": "",
        "expected_agent": "nana",
        "expected_inner_tool": "personal_create_schedule",
    },
    {
        "name": "group_slot",
        "request": "팀 회의 시간을 조율해줘",
        "member_replies": "민수: 2026-04-25 15:00 가능\n지아: 2026-04-25 15:00 가능",
        "expected_agent": "kana",
        "expected_inner_tool": "group_confirm_slot",
    },
]
extension_results = [run_supervisor_case(case) for case in extension_cases]

show_json([
    {
        "name": result["name"],
        "expected_agent": result["expected_agent"],
        "selected_agent": result["selected_agent"],
        "expected_inner_tool": result["expected_inner_tool"],
        "inner_tool_names": result["inner_tool_names"],
    }
    for result in extension_results
])

for result in extension_results:
    assert result["selected_agent"] == result["expected_agent"]
    assert result["expected_inner_tool"] in result["inner_tool_names"]
print("5주차 확장 예제 실행 통과")


## 6. 회고

개념 확인 질문:

1. supervisor가 직접 `personal_create_schedule`을 호출하지 않는 이유는 무엇인가?
2. `selected_agent`만 맞고 내부 tool이 틀리면 성공이라고 볼 수 있는가?
3. Golden Scenario는 눈으로 한 번 실행해 보는 것과 무엇이 다른가?

작은 응용 과제: 개인 일정처럼 보이지만 멤버 응답이 포함된 애매한 요청을 만들어 보고, supervisor가 어떤 agent를 선택하는지 토론한다.
